In [ ]:
!pip install -U transformers datasets accelerate scikit-learn gradio

In [ ]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())

In [ ]:
#loading datase
from datasets import load_dataset

dataset = load_dataset("ag_news")

print(dataset)

In [ ]:
#Tokenization
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

def tokenize(example):
    return tokenizer(example["text"], truncation=True, padding="max_length")

tokenized_datasets = dataset.map(tokenize, batched=True)

In [ ]:
#Prepare Dataset for Training
tokenized_datasets = tokenized_datasets.remove_columns(["text"])

In [ ]:
tokenized_datasets.set_format(
    "torch",
    columns=["input_ids", "attention_mask", "labels"]
)


In [ ]:
print(tokenized_datasets["train"].column_names)

In [ ]:
#Load Pretrained BERT Model
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=4
)


In [ ]:
#Training Setup
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=1,
    weight_decay=0.01,
    logging_dir="./logs"
)

In [ ]:
#Evaluation Metrics (Accuracy + F1)
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds, average="weighted")
    }

In [ ]:
# create trainer

from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"].shuffle(seed=42).select(range(20000)),
    eval_dataset=tokenized_datasets["test"].select(range(2000)),
    compute_metrics=compute_metrics
)

In [ ]:
trainer.train()

In [ ]:
trainer.evaluate()

In [ ]:
import numpy as np

predictions = trainer.predict(tokenized_datasets["test"].select(range(2000)))
y_pred = np.argmax(predictions.predictions, axis=1)
y_true = predictions.label_ids

In [ ]:
#craet confusion matrix
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_true, y_pred)
print(cm)

In [ ]:
#visulizing
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix - AG News Classification")
plt.show()

In [ ]:
labels = ["World", "Sports", "Business", "Sci/Tech"]

sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=labels,
            yticklabels=labels)

In [ ]:
trainer.save_model("news_classifier")
tokenizer.save_pretrained("news_classifier")

In [ ]:
import os
print(os.listdir("/content"))

In [ ]:
import shutil

shutil.make_archive(
    "/content/news_classifier",
    "zip",
    "/content/news_classifier"
)


In [ ]:
from google.colab import files

files.download("/content/news_classifier.zip")